# ATOM-SLM Dev — Stage 2 (better M1)

Stage 1 proved the pipeline but its GSM8K number was measured on a different
prompt than training. Stage 2 fixes the recipe **and** the measurement.

| | Stage 1 | Stage 2 |
|---|---|---|
| Data | 60k MetaMathQA rows | **100k rows, non-numeric answers dropped** |
| Epochs | 1 | **2** |
| Learning rate | 5e-5 | **1e-4 + 3% warmup** |
| Checkpoint kept | last | **best eval-loss checkpoint** |
| Eval prompt | 4-shot, `Answer:` cue | **0-shot, exact training prompt** |
| Eval size | 200 | **300** |
| Inspection | aggregates only | **loss curve + raw samples** |

**Before you run**

1. Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
2. Add-ons → Secrets → `HF_TOKEN` (a **write** token; the run pushes to the Hub).
3. Set `SMOKE_TEST = True` for a ~10 minute end-to-end check, then `False`.

Runtime for the real run: **~3–5 h** on a T4. Checkpoints go to the Hub every
500 steps, so a timeout costs minutes, not the whole run.

In [ ]:
import sys

import torch
import transformers

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0),
          "| devices:", torch.cuda.device_count())
print("transformers:", transformers.__version__)

In [ ]:
import os
import subprocess

from pathlib import Path

REPO_URL = "https://github.com/asaadzx/Atom_Slm.git"
REPO_DIR = Path("/kaggle/working/Atom_Slm")

if (REPO_DIR / ".git").exists():
    pulled = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True, text=True,
    )
    print(pulled.stdout or pulled.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

SRC_DIR = REPO_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

ENV = {
    **os.environ,
    "PYTHONPATH": f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}",
    "TOKENIZERS_PARALLELISM": "false",
    "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES", "0"),
}


def run(cmd: str, cwd: Path = REPO_DIR) -> None:
    print(f"$ {cmd}\n", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd), env=ENV)


print("repo:", REPO_DIR)

## 1 — Recipe

Only touch these values. `SMOKE_TEST` shrinks the run to 5 steps on 64 rows and
disables Hub pushes.

In [ ]:
import yaml

from pathlib import Path

RUN_NAME          = "m2a-smollm2-135m-metamath-2ep"
BASE_MODEL        = "HuggingFaceTB/SmolLM2-135M"
HF_REPO_ID        = "Asaadzx/atom-slm-math-135m-v2"
DATA              = "metamath"
MAX_SAMPLES       = 100_000
VAL_SAMPLES       = 1_000
MAX_LENGTH        = 1024
EPOCHS            = 2.0
PER_DEVICE_BATCH  = 8
GRAD_ACCUM        = 8
LEARNING_RATE     = 1e-4
SAVE_STEPS        = 500
EVAL_STEPS        = 500
EVAL_LIMIT        = 300
RUN_BASELINE_EVAL = True
SMOKE_TEST        = False

if SMOKE_TEST:
    MAX_SAMPLES, VAL_SAMPLES, MAX_LENGTH = 64, 16, 512
    SAVE_STEPS, EVAL_STEPS, EPOCHS = 5, 5, 1.0
    EVAL_LIMIT, RUN_BASELINE_EVAL = 50, False
    RUN_NAME = f"{RUN_NAME}-smoke"

config = {
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "method": "full",
    "precision": "fp16",
    "seed": 42,
    "data": {
        "name": DATA,
        "max_samples": MAX_SAMPLES,
        "val_samples": VAL_SAMPLES,
        "streaming": True,
        "require_numeric_answer": True,
    },
    "training": {
        "max_length": MAX_LENGTH,
        "epochs": EPOCHS,
        "per_device_batch_size": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler": "cosine",
        "warmup_ratio": 0.03,
        "optim": "adamw_torch",
        "gradient_checkpointing": True,
        "packing": True,
        "logging_steps": 20,
        "save_steps": SAVE_STEPS,
        "eval_steps": EVAL_STEPS,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
    },
    "hub": {
        "push_to_hub": not SMOKE_TEST,
        "repo_id": HF_REPO_ID,
        "private": False,
        "strategy": "every_save",
    },
}

CONFIG_PATH = Path(f"/kaggle/working/{RUN_NAME}.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
RUN_DIR = Path(f"/kaggle/working/runs/{RUN_NAME}")
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("SMOKE TEST" if SMOKE_TEST else "FULL RUN", "|", RUN_NAME)
print("config:", CONFIG_PATH)
print("effective batch:", PER_DEVICE_BATCH * GRAD_ACCUM,
      "| steps/epoch:", int(MAX_SAMPLES * 0.99) // (PER_DEVICE_BATCH * GRAD_ACCUM))

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login

    login(os.environ["HF_TOKEN"])
    CFG_HUB = config["hub"]
    print("Hub repo:", CFG_HUB["repo_id"], "| push:", CFG_HUB["push_to_hub"])
except Exception as exc:
    config["hub"]["push_to_hub"] = False
    CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    print("No HF token (", type(exc).__name__, ") - Hub push disabled.")

## 2 — Pre-flight: look at the data before spending GPU hours

This loads 24 rows, applies the real filter + formatting, and prints two training
transcripts with token counts. If the last line of a sample is not
`#### <number>`, stop and fix the data — training will not.

Expected: `Numeric-answer filter kept X/Y rows`, then samples ending in
`#### 18<|endoftext|>`.

In [ ]:
from transformers import AutoTokenizer

from atom_slm.data import build_sft_datasets

preflight = yaml.safe_load(CONFIG_PATH.read_text())
preflight["data"]["max_samples"] = 24
preflight["data"]["val_samples"] = 8

tokenizer = AutoTokenizer.from_pretrained(preflight["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
train, validation = build_sft_datasets(preflight, tokenizer)
print("chat template:", bool(getattr(tokenizer, "chat_template", None)))
print("train/validation rows:", len(train), len(validation))

for row in train.select(range(2)):
    text = row["text"]
    print("=" * 78)
    print(text[:1000])
    print("--- tokens:", len(tokenizer(text)["input_ids"]))

## 3 — Baseline GSM8K (0-shot, same prompt as training)

`EVAL_LIMIT = 300` takes ~10–12 minutes on a T4. Keep it: without a baseline the
tuned number means nothing.

In [ ]:
BASE_EVAL_PATH = RESULTS_DIR / "base-gsm8k.json"

if RUN_BASELINE_EVAL:
    run(
        f"{sys.executable} -m atom_slm.evaluate "
        f"--model {BASE_MODEL} --limit {EVAL_LIMIT} --num-fewshot 0 "
        f"--output {BASE_EVAL_PATH}"
    )
else:
    print("Skipped baseline eval.")

## 4 — Train

What changed vs Stage 1: 100k rows, 2 epochs, LR 1e-4, 3% warmup, numeric-only
answers, and the **best eval-loss checkpoint** is kept at the end.

In [ ]:
run(
    f"{sys.executable} -m atom_slm.train "
    f"--config {CONFIG_PATH} --output-dir {RUN_DIR}"
)

## 5 — Evaluate with the *same* prompt as training

0-shot, ending in `### Solution:`. This is the fix for Stage 1's misleading
1.00% — few-shot prompts change the task.

In [ ]:
import json

TUNED_EVAL_PATH = RESULTS_DIR / f"{RUN_NAME}-gsm8k.json"

run(
    f"{sys.executable} -m atom_slm.evaluate "
    f"--model {RUN_DIR} --limit {EVAL_LIMIT} --num-fewshot 0 "
    f"--output {TUNED_EVAL_PATH}"
)

reports = [str(p) for p in (BASE_EVAL_PATH, TUNED_EVAL_PATH) if p.exists()]
if reports:
    run(f"{sys.executable} -m atom_slm.evaluate --compare {' '.join(reports)}")

report = json.loads(TUNED_EVAL_PATH.read_text())
print(f"{report['prompt_style']} prompt | {report['num_fewshot']}-shot | "
      f"acc {100 * report['accuracy']:.2f}% | format {100 * report['format_rate']:.1f}% "
      f"| degenerate {100 * report['repetition_rate']:.1f}%")
for prediction in report["predictions"][:3]:
    print("-" * 78)
    print("gold:", prediction["gold"], "| pred:", prediction["prediction"],
          "| correct:", prediction["correct"])
    print(prediction["output"][:300].replace("\n", " "))

## 6 — Diagnose the run

Loss curve, mean weight change vs the base model, and two raw generations.
If samples here look wrong, the checkpoint is wrong — not the evaluator.

In [ ]:
run(f"{sys.executable} -m atom_slm.diagnose --run {RUN_DIR} --limit 2")

## 7 — Resume after a Kaggle timeout

`/kaggle/working` is wiped between sessions. Set `RESUME_FROM_HUB = True` in a
new session, rerun sections 1–3, then this cell and section 4: training resumes
from the last Hub checkpoint (`--resume auto`).

In [ ]:
RESUME_FROM_HUB = False

if RESUME_FROM_HUB:
    from huggingface_hub import snapshot_download

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_REPO_ID,
        local_dir=str(RUN_DIR),
        allow_patterns=["checkpoint-*/**"],
    )
    print("restored checkpoints:", sorted(p.name for p in RUN_DIR.glob("checkpoint-*")))
else:
    print("RESUME_FROM_HUB = False")

## Stage 2 acceptance criteria

| Signal | Target |
|---|---|
| Training | both epochs finish, `metrics.json` written |
| Loss | first eval > last eval, no NaN, no flatline |
| Format rate | **≥ 90%** |
| GSM8K | **> baseline**; 8–20% is a realistic Stage 2 result for 135M |
| Degenerate rate | **< 5%** |
| Samples | step-by-step arithmetic, then `#### <number>` |

If format is high but accuracy is low, the model learned the format but not the
math: move to **Stage 3** (OpenMathInstruct-2, 500k rows, `max_length` 2048) or
`SmolLM2-360M` with LoRA.

## Troubleshooting

| Symptom | Fix |
|---|---|
| `SFTTrainer does not support nn.DataParallel` | already pinned to GPU 0; if it appears, set `CUDA_VISIBLE_DEVICES=0` |
| CUDA OOM | `PER_DEVICE_BATCH = 4`, `GRAD_ACCUM = 16` |
| Loss NaN | fp16 only on T4; never bf16 |
| Hub 401/403 | `HF_TOKEN` must be a **write** token and attached to the notebook |
| `load_best_model_at_end` error | `SAVE_STEPS` must equal `EVAL_STEPS` |
| Run killed at 9–12 h | new session → `RESUME_FROM_HUB = True` → section 7 |